# Hands-On LAB 02 - Creando una Aplicación de Databricks

Entrenamiento Hands-on en la plataforma de Databricks con foco en la creación de aplicaciones interactivas usando Databricks Apps.
</br></br>

## Objetivos del Ejercicio

En este laboratorio aprenderás a:
* Crear una aplicación de Databricks desde cero usando Streamlit
* Conectar la aplicación a tablas de Unity Catalog
* Implementar operaciones CRUD (Crear, Leer, Actualizar) en datos
* Desplegar y compartir la aplicación con otros usuarios
* Aprovechar la simplicidad de la plataforma Databricks para desarrollo de aplicaciones

## Duración: ~30 minutos

## Prerrequisitos
* Haber completado el **Lab 01 - Importando los datos**
* Tener la tabla `opiniones` creada en Unity Catalog

## ¿Qué son las Databricks Apps?

**Databricks Apps** es una funcionalidad que permite crear y desplegar aplicaciones web interactivas directamente en la plataforma de Databricks, sin necesidad de infraestructura externa.

### Características principales:

* **Integración nativa**: Acceso directo a tablas de Unity Catalog, SQL Warehouses y otros recursos de Databricks
* **Framework Streamlit**: Usa el popular framework de Python para crear interfaces de usuario
* **Despliegue simplificado**: Un solo clic para desplegar y compartir aplicaciones
* **Seguridad integrada**: Hereda los permisos y controles de acceso de Databricks
* **Sin infraestructura**: No necesitas configurar servidores, contenedores o servicios externos

### Casos de uso comunes:

* Dashboards interactivos personalizados
* Herramientas de gestión de datos
* Aplicaciones de análisis self-service
* Interfaces para actualización de datos
* Prototipos de aplicaciones de ML

## Arquitectura de Nuestra Aplicación

Vamos a crear una aplicación de gestión de opiniones con tres funcionalidades principales:

```
┌─────────────────────────────────────────────────┐
│         Databricks App (Streamlit)              │
│                                                  │
│  ┌──────────┐  ┌──────────┐  ┌──────────┐     │
│  │ Ver Datos│  │  Agregar │  │  Editar  │     │
│  │          │  │  Opinion │  │  Opinion │     │
│  └────┬─────┘  └────┬─────┘  └────┬─────┘     │
│       │             │             │            │
│       └─────────────┴─────────────┘            │
│                     │                          │
└─────────────────────┼──────────────────────────┘
                      │
                      ▼
            ┌──────────────────┐
            │  SQL Warehouse   │
            └────────┬─────────┘
                     │
                     ▼
            ┌──────────────────┐
            │  Unity Catalog   │
            │                  │
            │  academia.ia     │
            │   .opiniones     │
            └──────────────────┘
```

### Componentes:

1. **Frontend (Streamlit)**: Interfaz de usuario con tres pestañas
2. **SQL Warehouse**: Motor de consultas para acceder a los datos
3. **Unity Catalog**: Almacenamiento de la tabla `opiniones`

## Paso 1: Crear una Nueva Aplicación

Vamos a crear nuestra primera Databricks App desde la interfaz de usuario.

### Instrucciones:

1. En la **barra lateral izquierda**, haz clic en el botón **Compute** 
2. En el menú desplegable, selecciona **App**
3. Se abrirá el asistente de creación de aplicaciones

![](../imagenes/create-app-step1.png)

### ¿Qué acabas de hacer?

Databricks está preparando un entorno aislado para tu aplicación, que incluye:
* Un espacio de trabajo para tu código
* Configuración de recursos computacionales
* Integración con Unity Catalog y SQL Warehouses

## Paso 2: Configurar los Ajustes de la Aplicación

Ahora vamos a configurar los parámetros básicos de nuestra aplicación.

### Configuración requerida:

1. **Nombre de la aplicación**: `Gestión de Opiniones - [tu-nombre]`
   * Ejemplo: `Gestión de Opiniones - Rico Martinez`
   * Este nombre será visible para otros usuarios

2. **Descripción** (opcional): `Aplicación para visualizar, agregar y editar opiniones de clientes`

3. **Tipo de aplicación**: Selecciona **Streamlit**
   * Streamlit es un framework de Python para crear aplicaciones web de forma sencilla

4. **SQL Warehouse**: Selecciona un warehouse disponible
   * Este será el motor de consultas que usará tu aplicación
   * Recomendado: Usar un Serverless SQL Warehouse para mejor rendimiento

5. Haz clic en **Create** para continuar

### ¿Por qué necesitamos un SQL Warehouse?

El SQL Warehouse es el motor computacional que ejecutará las consultas SQL de tu aplicación. Databricks Apps se conecta automáticamente a este warehouse para leer y escribir datos en Unity Catalog.

## Paso 3: Explorar la Estructura de la Aplicación

Después de crear la aplicación, Databricks genera automáticamente una estructura de archivos.

### Archivos generados:

```
mi-app/
├── app.py              # Código principal de la aplicación
├── app.yaml            # Configuración de la aplicación
└── requirements.txt    # Dependencias de Python
```

### Descripción de archivos:

* **app.py**: Contiene el código de Streamlit que define la interfaz y lógica de tu aplicación
* **app.yaml**: Archivo de configuración con variables de entorno (como el ID del SQL Warehouse)
* **requirements.txt**: Lista de paquetes de Python necesarios para tu aplicación

### Interfaz del editor:

El editor de Databricks Apps incluye:
* **Panel izquierdo**: Explorador de archivos
* **Panel central**: Editor de código con resaltado de sintaxis
* **Panel derecho**: Vista previa de la aplicación en tiempo real
* **Barra superior**: Botones para guardar, ejecutar y desplegar

## Paso 4: Entender el Archivo de Configuración (app.yaml)

El archivo `app.yaml` contiene la configuración esencial de tu aplicación.

### Contenido típico:

```yaml
command: ["streamlit", "run", "app.py", "--server.port=8080"]

env:
  - name: DATABRICKS_WAREHOUSE_ID
    value: "abc123def456"  # ID de tu SQL Warehouse
```

### Elementos importantes:

* **command**: Define cómo se ejecuta la aplicación (Streamlit en el puerto 8080)
* **env**: Variables de entorno disponibles en tu código
* **DATABRICKS_WAREHOUSE_ID**: ID del warehouse que configuraste en el Paso 2

### ¿Por qué es importante?

Este archivo permite que tu aplicación:
1. Se conecte automáticamente al SQL Warehouse correcto
2. Herede las credenciales de autenticación de Databricks
3. Acceda a Unity Catalog sin configuración adicional

**Nota**: No necesitas modificar este archivo manualmente; Databricks lo gestiona automáticamente.

## Paso 5: Crear la Estructura Básica de app.py

Ahora vamos a escribir el código de nuestra aplicación. Reemplaza el contenido de `app.py` con el siguiente código.

### Parte 1: Importaciones y Configuración

Primero, importamos las librerías necesarias y configuramos la conexión a Databricks:

```python
import os
from databricks import sql
from databricks.sdk.core import Config
import streamlit as st
import pandas as pd
from datetime import date

# Verificar que la variable de entorno esté configurada
assert os.getenv('DATABRICKS_WAREHOUSE_ID'), "DATABRICKS_WAREHOUSE_ID must be set in app.yaml."
```

### ¿Qué hace este código?

* **databricks.sql**: SDK para conectarse a SQL Warehouses
* **streamlit**: Framework para crear la interfaz de usuario
* **pandas**: Manipulación de datos en formato DataFrame
* **Config()**: Obtiene automáticamente las credenciales de autenticación
* **assert**: Verifica que el warehouse ID esté configurado correctamente

## Paso 6: Crear Funciones para Ejecutar SQL

Ahora definimos dos funciones para interactuar con la base de datos.

### Función para consultas SELECT:

```python
def sqlQuery(query: str) -> pd.DataFrame:
    """Ejecuta una consulta SQL y retorna un DataFrame de pandas"""
    cfg = Config()  # Obtiene credenciales automáticamente
    with sql.connect(
        server_hostname=cfg.host,
        http_path=f"/sql/1.0/warehouses/{os.getenv('DATABRICKS_WAREHOUSE_ID')}",
        credentials_provider=lambda: cfg.authenticate
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
            result = cursor.fetchall_arrow()
            if result:
                return result.to_pandas()
            return pd.DataFrame()
```

### Función para INSERT/UPDATE:

```python
def sqlExecute(query: str):
    """Ejecuta una sentencia SQL sin retornar resultados (INSERT/UPDATE)"""
    cfg = Config()
    with sql.connect(
        server_hostname=cfg.host,
        http_path=f"/sql/1.0/warehouses/{os.getenv('DATABRICKS_WAREHOUSE_ID')}",
        credentials_provider=lambda: cfg.authenticate
    ) as connection:
        with connection.cursor() as cursor:
            cursor.execute(query)
```

### Ventajas de este enfoque:

* **Autenticación automática**: No necesitas gestionar tokens o contraseñas
* **Reutilizable**: Puedes usar estas funciones en toda tu aplicación
* **Seguro**: Usa las credenciales del usuario que ejecuta la app

## Paso 7: Configurar Streamlit y Cargar Datos

Ahora configuramos la interfaz de Streamlit y creamos una función para cargar datos.

### Configuración de página:

```python
st.set_page_config(layout="wide")
```

Esto hace que la aplicación use todo el ancho de la pantalla.

### Función para cargar datos con caché:

```python
@st.cache_data(ttl=30)  # Cache por 30 segundos
def getData():
    return sqlQuery("SELECT * FROM academia.ia.opiniones ORDER BY fecha DESC, id_opinion DESC")
```

### ¿Qué es el caché?

* **@st.cache_data**: Decorador de Streamlit que almacena resultados en memoria
* **ttl=30**: Los datos se actualizan cada 30 segundos
* **Beneficio**: Evita consultas repetidas a la base de datos, mejorando el rendimiento

### Título de la aplicación:

```python
st.header("Sistema de Gestión de Opiniones")
```

**Importante**: Reemplaza `academia.ia.opiniones` con el nombre completo de tu tabla (catalog.schema.table) que creaste en el Lab 01.

## Paso 8: Crear la Estructura de Pestañas

Streamlit permite organizar la interfaz en pestañas para mejor experiencia de usuario.

### Crear tres pestañas:

```python
tab1, tab2, tab3 = st.tabs(["📊 Ver Datos", "➕ Agregar Opinion", "✏️ Editar Opinion"])
```

### Pestaña 1: Ver Datos

```python
with tab1:
    st.subheader("Todas las Opiniones")
    data = getData()
    st.dataframe(data=data, height=600, use_container_width=True)
    st.info(f"Total de registros: {len(data)}")
```

### ¿Qué hace este código?

* **st.tabs()**: Crea pestañas interactivas con iconos emoji
* **with tab1**: Define el contenido de la primera pestaña
* **st.dataframe()**: Muestra los datos en una tabla interactiva
  * `height=600`: Altura de la tabla en píxeles
  * `use_container_width=True`: Usa todo el ancho disponible
* **st.info()**: Muestra un mensaje informativo con el total de registros

### Resultado:

Los usuarios verán una tabla interactiva con todas las opiniones, ordenadas por fecha y ID.

## Paso 9: Crear el Formulario para Agregar Opiniones

Ahora implementamos la funcionalidad para agregar nuevas opiniones.

### Pestaña 2: Agregar Opinion

```python
with tab2:
    st.subheader("Agregar Nueva Opinion")
    with st.form("add_form"):
        col1, col2 = st.columns(2)
        with col1:
            new_fecha = st.date_input("Fecha", value=date.today())
            new_id_opinion = st.number_input("ID Opinion", min_value=1, step=1)
        with col2:
            new_id_cliente = st.number_input("ID Cliente", min_value=1, step=1)
        
        new_opinion = st.text_area("Opinion", height=150)
        
        submitted = st.form_submit_button("Agregar Opinion")
        if submitted:
            if new_opinion.strip():
                try:
                    st.cache_data.clear()
                    insert_query = f"""
                    INSERT INTO academia.ia.opiniones 
                    (fecha, id_opinion, id_cliente, opinion)
                    VALUES ('{new_fecha}', {new_id_opinion}, {new_id_cliente}, '{new_opinion.replace("'", "''")}")
                    """
                    sqlExecute(insert_query)
                    st.success("✅ Opinion agregada exitosamente!")
                    st.rerun()
                except Exception as e:
                    st.error(f"❌ Error al agregar opinion: {str(e)}")
            else:
                st.warning("⚠️ Por favor ingresa el texto de la opinion.")
```

### Elementos clave:

* **st.form()**: Agrupa inputs y envía todos juntos
* **st.columns(2)**: Divide el espacio en dos columnas
* **st.date_input()**: Selector de fecha
* **st.number_input()**: Campo numérico con validación
* **st.text_area()**: Campo de texto multilínea
* **st.form_submit_button()**: Botón que envía el formulario
* **st.cache_data.clear()**: Limpia el caché para mostrar datos actualizados
* **st.rerun()**: Recarga la aplicación para mostrar los cambios

**Nota de seguridad**: El código usa `.replace("'", "''")` para escapar comillas simples y prevenir errores de SQL.

## Paso 10: Crear el Formulario para Editar Opiniones

Finalmente, implementamos la funcionalidad para editar opiniones existentes.

### Pestaña 3: Editar Opinion

```python
with tab3:
    st.subheader("Editar Opinion Existente")
    data = getData()
    
    if len(data) > 0:
        # Crear un dropdown con las opiniones
        data['display'] = data.apply(lambda row: f"ID {row['id_opinion']} - Cliente {row['id_cliente']} - {row['fecha']}", axis=1)
        selected_display = st.selectbox("Selecciona la opinion a editar", data['display'].tolist())
        
        if selected_display:
            selected_row = data[data['display'] == selected_display].iloc[0]
            
            with st.form("edit_form"):
                st.write(f"**Fecha:** {selected_row['fecha']}")
                st.write(f"**ID Opinion:** {selected_row['id_opinion']}")
                st.write(f"**ID Cliente:** {selected_row['id_cliente']}")
                
                st.write("**Opinion Actual:**")
                st.info(selected_row['opinion'])
                
                updated_opinion = st.text_area("Nueva Opinion", value=selected_row['opinion'], height=150)
                
                submitted_edit = st.form_submit_button("Actualizar Opinion")
                if submitted_edit:
                    if updated_opinion.strip():
                        try:
                            st.cache_data.clear()
                            update_query = f"""
                            UPDATE academia.ia.opiniones 
                            SET opinion = '{updated_opinion.replace("'", "''")}'
                            WHERE id_opinion = {selected_row['id_opinion']} 
                            AND id_cliente = {selected_row['id_cliente']}
                            AND fecha = '{selected_row['fecha']}'
                            """
                            sqlExecute(update_query)
                            st.success("✅ Opinion actualizada exitosamente!")
                            st.rerun()
                        except Exception as e:
                            st.error(f"❌ Error al actualizar opinion: {str(e)}")
                    else:
                        st.warning("⚠️ La opinion no puede estar vacía.")
    else:
        st.info("No hay opiniones disponibles para editar. ¡Agrega algunas primero!")
```

### Elementos clave:

* **st.selectbox()**: Dropdown para seleccionar una opinion
* **data['display']**: Columna calculada para mostrar información legible
* **selected_row**: Fila seleccionada del DataFrame
* **UPDATE query**: Actualiza solo la columna `opinion` del registro seleccionado

## Paso 11: Probar la Aplicación Localmente

Antes de desplegar, vamos a probar la aplicación en modo de desarrollo.

### Instrucciones:

1. En la parte superior del editor, haz clic en el botón **▶ Run**
2. Espera unos segundos mientras Databricks inicia el servidor de Streamlit
3. La aplicación se abrirá en el **panel derecho** del editor

### ¿Qué deberías ver?

* **Pestaña "Ver Datos"**: Tabla con todas las opiniones de tu base de datos
* **Pestaña "Agregar Opinion"**: Formulario para insertar nuevos registros
* **Pestaña "Editar Opinion"**: Dropdown y formulario para modificar opiniones

### Pruebas recomendadas:

1. **Ver datos**: Verifica que se muestren las opiniones del Lab 01
2. **Agregar**: Crea una nueva opinion de prueba
   * Fecha: Hoy
   * ID Opinion: 999
   * ID Cliente: 1
   * Opinion: "Esta es una prueba de la aplicación"
3. **Editar**: Selecciona la opinion que acabas de crear y modifica el texto
4. **Verificar**: Vuelve a la pestaña "Ver Datos" para confirmar los cambios

### Solución de problemas:

* **Error de conexión**: Verifica que el SQL Warehouse esté activo
* **Tabla no encontrada**: Confirma el nombre completo de la tabla en `getData()`
* **Error de permisos**: Asegúrate de tener permisos de lectura/escritura en la tabla

## Paso 12: Desplegar la Aplicación

Una vez que la aplicación funciona correctamente, es hora de desplegarla para que otros usuarios puedan acceder.

### Instrucciones de despliegue:

1. En la parte superior del editor, haz clic en el botón **Deploy** (azul)
2. Databricks mostrará un diálogo de confirmación
3. Revisa la configuración:
   * **Nombre**: Gestión de Opiniones - [tu-nombre]
   * **SQL Warehouse**: El warehouse configurado
   * **Permisos**: Por defecto, solo tú tienes acceso
4. Haz clic en **Confirm Deploy**

### ¿Qué sucede durante el despliegue?

Databricks automáticamente:
1. **Empaqueta** tu código y dependencias
2. **Crea** un contenedor con el entorno de ejecución
3. **Inicia** el servidor de Streamlit
4. **Genera** una URL pública para acceder a la aplicación
5. **Configura** la autenticación y permisos

### Tiempo de despliegue:

* Primera vez: ~2-3 minutos
* Actualizaciones posteriores: ~30-60 segundos

### Monitoreo del despliegue:

Puedes ver el progreso en la parte inferior del editor:
* **Building**: Empaquetando la aplicación
* **Starting**: Iniciando el servidor
* **Running**: Aplicación lista y accesible

**Nota**: Durante el despliegue, la aplicación no estará disponible temporalmente.

## Paso 13: Acceder y Compartir la Aplicación

Ahora que la aplicación está desplegada, puedes acceder a ella y compartirla con otros usuarios.

### Acceder a tu aplicación:

1. Una vez completado el despliegue, verás un mensaje: **"App is running"**
2. Haz clic en el botón **Open App** o en la URL generada
3. La aplicación se abrirá en una nueva pestaña del navegador
4. La URL tendrá el formato: `https://<workspace>.cloud.databricks.com/apps/<app-id>`

### Compartir con otros usuarios:

1. En el editor de la aplicación, haz clic en el botón **Share** (esquina superior derecha)
2. Se abrirá el diálogo de permisos
3. Opciones de compartir:
   * **Usuarios específicos**: Agrega emails o nombres de usuario
   * **Grupos**: Comparte con grupos de tu organización
   * **Todos los usuarios**: Hace la app pública en tu workspace
4. Selecciona el nivel de permiso:
   * **Can view**: Solo pueden usar la aplicación
   * **Can edit**: Pueden modificar el código
5. Haz clic en **Add** y luego **Save**

### Gestión de permisos:

* Los usuarios heredan los permisos de Unity Catalog
* Si un usuario no tiene acceso a la tabla `opiniones`, verá un error
* Puedes revocar acceso en cualquier momento desde el diálogo de permisos

### URL permanente:

La URL de tu aplicación es permanente y puedes:
* Compartirla por email o chat
* Agregarla a favoritos
* Incrustarla en dashboards o portales internos

## Paso 14: Actualizar y Redesplegar la Aplicación

A medida que tu aplicación evoluciona, necesitarás hacer cambios y redesplegarla.

### Proceso de actualización:

1. **Editar el código**: Modifica `app.py` según tus necesidades
2. **Probar localmente**: Usa el botón **Run** para verificar los cambios
3. **Redesplegar**: Haz clic en **Deploy** nuevamente
4. **Confirmar**: Databricks actualizará la aplicación en producción

### Ejemplos de mejoras que puedes hacer:

#### Agregar filtros:
```python
# En la pestaña "Ver Datos"
fecha_inicio = st.date_input("Fecha inicio")
fecha_fin = st.date_input("Fecha fin")
data_filtrada = data[(data['fecha'] >= fecha_inicio) & (data['fecha'] <= fecha_fin)]
st.dataframe(data_filtrada)
```

#### Agregar gráficos:
```python
import plotly.express as px

# Gráfico de opiniones por cliente
fig = px.bar(data.groupby('id_cliente').size().reset_index(name='count'), 
             x='id_cliente', y='count', title='Opiniones por Cliente')
st.plotly_chart(fig)
```

#### Agregar búsqueda:
```python
busqueda = st.text_input("Buscar en opiniones")
if busqueda:
    data_filtrada = data[data['opinion'].str.contains(busqueda, case=False)]
    st.dataframe(data_filtrada)
```

### Versionado:

Databricks mantiene un historial de despliegues:
* Puedes ver versiones anteriores en la sección **Deployments**
* Puedes revertir a una versión anterior si es necesario
* Cada despliegue incluye timestamp y usuario que lo realizó

## Paso 15: Monitorear y Solucionar Problemas

Databricks proporciona herramientas para monitorear el rendimiento y diagnosticar problemas.

### Panel de monitoreo:

1. En el editor de la aplicación, haz clic en la pestaña **Monitoring**
2. Verás métricas en tiempo real:
   * **Usuarios activos**: Cuántos usuarios están usando la app
   * **Requests**: Número de peticiones por minuto
   * **Latencia**: Tiempo de respuesta promedio
   * **Errores**: Tasa de errores y excepciones

### Logs de la aplicación:

1. Haz clic en la pestaña **Logs**
2. Verás los logs de Streamlit y Python
3. Útil para:
   * Depurar errores
   * Ver consultas SQL ejecutadas
   * Identificar problemas de rendimiento

### Problemas comunes y soluciones:

#### 1. Aplicación lenta:
* **Causa**: Consultas SQL sin optimizar
* **Solución**: Agrega índices, limita resultados, usa caché

#### 2. Errores de conexión:
* **Causa**: SQL Warehouse pausado o sin capacidad
* **Solución**: Verifica el estado del warehouse, aumenta el tamaño si es necesario

#### 3. Datos no actualizados:
* **Causa**: Caché de Streamlit
* **Solución**: Reduce el `ttl` en `@st.cache_data` o agrega un botón de refresh

#### 4. Errores de permisos:
* **Causa**: Usuario sin acceso a la tabla
* **Solución**: Otorga permisos en Unity Catalog con `GRANT SELECT, INSERT, UPDATE`

### Mejores prácticas:

* Monitorea regularmente el uso de la aplicación
* Configura alertas para errores críticos
* Mantén los logs por al menos 30 días
* Documenta cambios importantes en el código

## Conclusión

¡Felicitaciones! Has creado y desplegado exitosamente tu primera Databricks App. 🎉

### Lo que aprendiste:

✅ Crear una aplicación de Databricks desde cero
✅ Conectar la aplicación a Unity Catalog usando SQL Warehouses
✅ Implementar operaciones CRUD (Crear, Leer, Actualizar) en datos
✅ Usar Streamlit para crear interfaces interactivas
✅ Desplegar y compartir aplicaciones con otros usuarios
✅ Monitorear y solucionar problemas en producción

### Ventajas de Databricks Apps:

* **Simplicidad**: Sin necesidad de configurar infraestructura externa
* **Integración**: Acceso nativo a todos los recursos de Databricks
* **Seguridad**: Hereda permisos y controles de acceso
* **Escalabilidad**: Se adapta automáticamente a la carga de usuarios
* **Rapidez**: De idea a producción en minutos, no días

### Próximos pasos:

1. **Mejora la aplicación**:
   * Agrega gráficos y visualizaciones
   * Implementa filtros y búsqueda avanzada
   * Añade validaciones de datos más robustas

2. **Integra con IA**:
   * Usa funciones SQL AI para analizar sentimientos en opiniones
   * Implementa clasificación automática de opiniones
   * Genera resúmenes con modelos de lenguaje

3. **Explora más funcionalidades**:
   * Conecta múltiples tablas
   * Crea dashboards interactivos
   * Implementa autenticación personalizada

### Recursos adicionales:

* [Documentación de Databricks Apps](https://apps-cookbook.dev/)
* [Streamlit Documentation](https://docs.streamlit.io/)
* [Unity Catalog Best Practices](https://docs.databricks.com/data-governance/unity-catalog/)

---

**¿Preguntas o problemas?** Consulta los logs de la aplicación o contacta al equipo de soporte de Databricks.